In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#!git clone https://github.com/recsyspolimi/RecSys_Course_AT_PoliMi

os.chdir("/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/RecSys_Course_AT_PoliMi")

!pwd

#!python run_compile_all_cython.py

In [ ]:
import os
import time 
import torch
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.sparse as sps
import matplotlib.pyplot as pyplot
%matplotlib inline

from sklearn.model_selection import KFold
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from skopt.space import Real, Integer, Categorical
from Evaluation.Evaluator import EvaluatorHoldout
from HyperparameterTuning.SearchBayesianSkopt import SearchBayesianSkopt
from Recommenders.Neural.MultVAE_PyTorch_Recommender import MultVAERecommender_PyTorch
from Recommenders.Neural.MultVAE_PyTorch_Recommender import MultVAERecommender_PyTorch_OptimizerMask

In [ ]:
df_train = pd.read_csv("data_train.csv")
df_test_user = pd.read_csv("data_target_users_test.csv")

In [ ]:
def split_train_in_five_percentage_global_sample(URM_all, train_percentages):
    """
    The function splits an URM in five matrices based on provided percentages.
    :param URM_all: The full URM matrix
    :param train_percentages: A list of percentages (must sum to 1.0)
    :return: A list of 5 sparse matrices
    """

    import numpy as np
    from scipy.sparse import coo_matrix
    from Data_manager.IncrementalSparseMatrix import IncrementalSparseMatrix

    assert len(train_percentages) == 5, "You must provide exactly 5 percentages."
    assert abs(sum(train_percentages) - 1.0) < 1e-6, "Percentages must sum to 1.0."

    num_users, num_items = URM_all.shape

    # Builders for each of the 5 matrices
    builders = [
        IncrementalSparseMatrix(n_rows=num_users, n_cols=num_items, auto_create_col_mapper=False, auto_create_row_mapper=False)
        for _ in range(5)
    ]

    URM_all_coo = coo_matrix(URM_all)

    # Shuffle indices
    indices_for_sampling = np.arange(URM_all.nnz, dtype=np.int32)
    np.random.shuffle(indices_for_sampling)

    # Calculate the number of interactions for each split
    split_sizes = [int(URM_all.nnz * percentage) for percentage in train_percentages]
    cumulative_sizes = np.cumsum(split_sizes)

    # Divide the indices into 5 groups
    indices_splits = [
        indices_for_sampling[cumulative_sizes[i - 1]:cumulative_sizes[i]] if i > 0 else indices_for_sampling[:cumulative_sizes[i]]
        for i in range(5)
    ]

    # Populate the builders
    for i, builder in enumerate(builders):
        builder.add_data_lists(
            URM_all_coo.row[indices_splits[i]],
            URM_all_coo.col[indices_splits[i]],
            URM_all_coo.data[indices_splits[i]],
        )

    # Convert to sparse matrices
    sparse_matrices = [builder.get_SparseMatrix() for builder in builders]

    # Ensure all outputs are in csr_matrix format
    sparse_matrices = [sp.csr_matrix(matrix) for matrix in sparse_matrices]

    return sparse_matrices

In [ ]:
from scipy.sparse import coo_matrix

#valore 1 per ogni coppia (row, col)
data = [1] * len(df_train)
df_train["row"] = df_train["row"].astype(int)
df_train["col"] = df_train["col"].astype(int)

# matrice COO
URM_all = sp.csr_matrix((data, (df_train["row"], df_train["col"])))

In [ ]:
train_percentages = [0.2, 0.2, 0.2, 0.2, 0.2]  # Cinque parti uguali

URM_parts = split_train_in_five_percentage_global_sample(URM_all, train_percentages)
URM_parts

In [ ]:
import time 

class SaveResults(object):
    
    def __init__(self):
        self.results_df = pd.DataFrame(columns=["result", "train_time (min)"])
    
    def __call__(self, optuna_study, optuna_trial):
        hyperparam_dict = optuna_trial.params.copy()
        hyperparam_dict["result"] = optuna_trial.values[0]
        
        # Retrieve the optimal number of epochs and training time from the "user attributes" of the trial
        #hyperparam_dict["epochs"] = optuna_trial.user_attrs["epochs"]
        hyperparam_dict["train_time (min)"] = optuna_trial.user_attrs["train_time (min)"]
        
        self.results_df.loc[len(self.results_df)] = hyperparam_dict
        
        


def objective_function_multvae_light(optuna_trial):
    start_time = time.time()
    # Parametri "Light" per testare in locale
    epochs = 5  # Solo per testare se il loop funziona
    batch_size = 32  # Valore basso = meno RAM occupata
    encoding_size = optuna_trial.suggest_int("encoding_size", 20, 50) # Collo di bottiglia stretto
    learning_rate = optuna_trial.suggest_float("learning_rate", 1e-4, 1e-3, log=True)
    
    # In locale facciamo 1 solo fold invece di 5 per velocità
    scores = []
    for i in range(1): 
        URM_train = sum(URM_parts[j] for j in range(len(URM_parts)) if j != i)
        evaluator_validation = EvaluatorHoldout(URM_parts[i], cutoff_list=[20])
        
        # CORREZIONE 1: Nome parametro 'use_gpu' e classe 'OptimizerMask'
        recommender_instance = MultVAERecommender_PyTorch_OptimizerMask(URM_train, use_gpu=False)
        
        # CORREZIONE 2: Passa gli iperparametri corretti per la versione OptimizerMask
        recommender_instance.fit(
            epochs = 5,
            batch_size = 32,
            learning_rate = optuna_trial.suggest_float("learning_rate", 1e-4, 1e-3, log=True),
            l2_reg = optuna_trial.suggest_float("l2_reg", 1e-5, 1e-2, log=True),
            dropout = 0.3,
            
            # Parametri gestiti dalla classe OptimizerMask
            encoding_size = optuna_trial.suggest_int("encoding_size", 20, 50),
            next_layer_size_multiplier = 1.5,
            sgd_mode = "adam",
            
            # Early Stopping
            evaluator_object = evaluator_validation,
            validation_every_n = 1,
            lower_validations_allowed = 2,
            validation_metric = "RECALL"
        )
        
        result, _ = evaluator_validation.evaluateRecommender(recommender_instance)
        scores.append(result.loc[20]["RECALL"])

    train_time_min = (time.time() - start_time) / 60
    optuna_trial.set_user_attr("train_time (min)", train_time_min)
        
    return sum(scores) / len(scores)

In [ ]:
import numpy as np
import torch
import math
import torch.nn.functional as f

# 1. Definizione della versione corretta di _run_epoch (Fix indici Scipy)
def patched_run_epoch(self, num_epoch):
    num_batches_per_epoch = math.ceil(len(self.warm_user_ids) / self.batch_size)
    self._model.train()
    epoch_loss = 0

    for _ in range(num_batches_per_epoch):
        self._optimizer.zero_grad()

        # FIX: u rimane un array numpy per indicizzare la matrice scipy URM_train
        u_idx = np.random.choice(self.warm_user_ids, size=self.batch_size)
        user_batch_tensor = self.URM_train[u_idx]

        # Trasferimento sicuro al device (CPU/GPU)
        user_batch_tensor = torch.sparse_csr_tensor(user_batch_tensor.indptr,
                                                    user_batch_tensor.indices,
                                                    user_batch_tensor.data,
                                                    size=user_batch_tensor.shape, 
                                                    dtype=torch.float32, 
                                                    device=self.device).to_dense()

        logits, KL, mu_q, std_q, epsilon, sampled_z = self._model.forward(user_batch_tensor)

        log_softmax_var = f.log_softmax(logits, dim=1)
        neg_ll = - torch.mean(torch.sum(log_softmax_var * user_batch_tensor, dim=1))
        l2_reg = self._model.get_l2_reg()

        anneal = min(self.anneal_cap, 1. * self.update_count / self.total_anneal_steps) if self.total_anneal_steps > 0 else self.anneal_cap

        loss = neg_ll + anneal * KL + l2_reg * self.l2_reg
        self.update_count += 1
        loss.backward()
        epoch_loss += loss.item()
        self._optimizer.step()

    self._print("Loss {:.2E}".format(epoch_loss))
    self._model.eval()

# 2. Definizione della versione corretta di _compute_item_score
def patched_compute_item_score(self, user_id_array, items_to_compute = None):
    # FIX: user_id_array viene usato direttamente come numpy
    user_batch_tensor = self.URM_train[user_id_array]
    user_batch_tensor = torch.sparse_csr_tensor(user_batch_tensor.indptr,
                                                user_batch_tensor.indices,
                                                user_batch_tensor.data,
                                                size=user_batch_tensor.shape, 
                                                dtype=torch.float32,
                                                device=self.device).to_dense()

    with torch.no_grad():
        self._model.eval()
        logits, _, _, _, _, _ = self._model.forward(user_batch_tensor)

    item_scores_to_compute = logits.cpu().detach().numpy()

    if items_to_compute is not None:
        item_scores = - np.ones((len(user_id_array), self.n_items)) * np.inf
        item_scores[:, items_to_compute] = item_scores_to_compute[:, items_to_compute]
    else:
        item_scores = item_scores_to_compute

    return item_scores

# 3. Applicazione forzata dei metodi corretti alla classe importata
MultVAERecommender_PyTorch._run_epoch = patched_run_epoch
MultVAERecommender_PyTorch._compute_item_score = patched_compute_item_score

In [ ]:
import optuna
optuna_study = optuna.create_study(direction="maximize")
        
save_results = SaveResults()
        
optuna_study.optimize(objective_function_multvae_light,
                      callbacks=[save_results],
                      n_trials = 50)

In [ ]:
optuna_study.best_trial.params

In [ ]:
save_results.results_df

In [ ]:
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_param_importances


plot_param_importances(optuna_study)

In [ ]:
#plot_parallel_coordinate(optuna_study, params=["topK", "shrink", "tversky_alpha", "tversky_beta"])

In [ ]:
best_index = save_results.results_df["result"].idxmax()
best_hyperparams = save_results.results_df.loc[best_index].to_dict()

del best_hyperparams["result"]
del best_hyperparams["train_time (min)"]
best_hyperparams

